In [19]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
import joblib
import os
from datetime import datetime

# Preprocessing
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer

# Models
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.neural_network import MLPRegressor
from openpyxl import load_workbook

# Model selection
from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    KFold,
)

# Metrics
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Feature importance
from sklearn.inspection import permutation_importance

In [20]:
# =============================================================================
# SECTION 1: LOAD DATA
# =============================================================================

CONFIG = {
    "input": "Outputs/clean_property_data.parquet",
    "test_size": 0.2,
    "random_state": 42,
    "cv_folds": 5,
    "n_jobs": 1,  # keep at 1 to avoid memory crashes on large data
}

print("Loading dataset...")
df = pd.read_parquet(CONFIG["input"])
print(f"  Rows: {len(df):,}, Columns: {len(df.columns)}")
print(f"  Columns: {df.columns.tolist()}")

df = df.sample(n=100_000, random_state=42) 

Loading dataset...
  Rows: 4,775,719, Columns: 29
  Columns: ['price', 'property_type_x', 'new_build', 'duration', 'address_key_short', 'total_floor_area', 'current_energy_efficiency', 'current_energy_rating', 'number_habitable_rooms', 'tenure', 'construction_age_band', 'built_form', 'main_fuel', 'postcode', 'property_type', 'exact_lat', 'exact_lon', 'dist_primary_km', 'dist_secondary_km', 'dist_rail_km', 'rail_within_1km', 'rail_within_5km', 'dist_metro_km', 'metro_within_1km', 'dist_airport_km', 'dist_coast_km', 'dist_town_km', 'sale_time', 'construction_year']


In [21]:
# =============================================================================
# SECTION 2: DEFINE FEATURES AND TARGET
# =============================================================================

target = "price"

numeric_features = [
    # EPC features
    "total_floor_area",           # floor area in m²
    "current_energy_efficiency",  # numeric efficiency score
    "number_habitable_rooms",     # number of rooms

    # Location — exact coordinates
    "exact_lat",                  # property latitude
    "exact_lon",                  # property longitude

    # Spatial — school distances
    "dist_primary_km",            # distance to nearest primary school
    "dist_secondary_km",          # distance to nearest secondary school

    # Spatial — transport distances and density
    "dist_rail_km",               # distance to nearest rail station
    "rail_within_1km",            # number of rail stations within 1km
    "rail_within_5km",            # number of rail stations within 5km
    "dist_metro_km",              # distance to nearest metro/underground
    "metro_within_1km",           # number of metro stations within 1km
    "dist_airport_km",            # distance to nearest airport

    # Spatial — geography
    "dist_coast_km",              # distance to nearest coastline
    "dist_town_km",               # distance to nearest major town (ONS 75k+)

    # Date
    #"sale_year",                  # year of sale
    #"sale_month",                 # month of sale
    "sale_time",
    "construction_year",            # year of construction
]

categorical_features = [
    # Land Registry
    "property_type_x",            # D=Detached, S=Semi, T=Terraced, F=Flat
    "new_build",                  # Y/N
    "duration",                   # F=Freehold, L=Leasehold

    # EPC
    "current_energy_rating",      # A-G energy rating
    "tenure",                     # owner-occupied, rented, etc.
    "built_form",                 # Detached, Semi-Detached, Mid-Terrace, etc.
    "construction_age_band",      # e.g. 1900-1929, 1967-1975
    "main_fuel",                  # mains gas, electricity, etc.
]

# Removed from previous version:
# - property_type_y: overlaps with property_type_x and built_form
# - transaction_type: describes why EPC was created, not a property attribute
# - mains_gas_flag: 43% NaN, redundant with main_fuel
# - binary_features: new_build moved to categorical (Y/N works with OneHot)

all_features = numeric_features + categorical_features

# Verify all features exist
missing = [f for f in all_features if f not in df.columns]
if missing:
    print(f"  WARNING: missing columns: {missing}")
    numeric_features = [f for f in numeric_features if f in df.columns]
    categorical_features = [f for f in categorical_features if f in df.columns]
    all_features = numeric_features + categorical_features

X = df[all_features]
y = df[target]

print(f"\n  Features: {len(all_features)}")
print(f"    Numeric:     {len(numeric_features)}")
print(f"    Categorical: {len(categorical_features)}")
print(f"  Target: {target} (mean=£{y.mean():,.0f}, median=£{y.median():,.0f})")


  Features: 25
    Numeric:     17
    Categorical: 8
  Target: price (mean=£335,899, median=£270,000)


In [22]:
# =============================================================================
# SECTION 3: TRAIN/TEST SPLIT
# =============================================================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=CONFIG["test_size"],
    random_state=CONFIG["random_state"],
)

print(f"  X_train: {X_train.shape}")
print(f"  X_test:  {X_test.shape}")

  X_train: (80000, 25)
  X_test:  (20000, 25)


In [23]:
# =============================================================================
# SECTION 4: PREPROCESSOR
# =============================================================================

# Numeric: fill any remaining NaN with median, then scale
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

# Categorical: fill any remaining NaN with "Unknown", then one-hot encode
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

In [24]:
# =============================================================================
# SECTION 5: BUILD PIPELINES
# =============================================================================

pipe_ridge = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", Ridge())
])

pipe_lasso = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", Lasso(max_iter=1000))  # up to 5000 for final training
])

pipe_rf = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(random_state=42, n_jobs=-1))
])

pipe_xgb = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", XGBRegressor(random_state=42, n_jobs=-1, tree_method="hist"))
])

pipe_mlp = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", MLPRegressor(random_state=42, max_iter=500, early_stopping=True))
])

In [ ]:
# =============================================================================
# SECTION 6: PARAM GRIDS
# =============================================================================

param_grids = {
    "Ridge": {
        "pipeline": pipe_ridge,
        "params": {
            "regressor__alpha": [1.0, 10.0, 100.0, 150.0, 200.0],
        }
    },
    "XGBoost": {
        "pipeline": pipe_xgb,
        "params": {
            "regressor__n_estimators": [200, 300, 500],
            "regressor__max_depth": [6, 8, 10],
            "regressor__learning_rate": [0.05, 0.1],
            "regressor__subsample": [0.8],
            "regressor__colsample_bytree": [0.8],
        }
    },
    "RandomForest": {
        "pipeline": pipe_rf,
        "params": {
            "regressor__n_estimators": [100, 200],
            "regressor__max_depth": [10, 20, None],
            "regressor__min_samples_leaf": [5, 10],
        }
    },
}

In [26]:
# =============================================================================
# SECTION 7: QUICK TEST — Ridge baseline without GridSearch
# =============================================================================

pipe_quick = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", Ridge(alpha=100.0))
])

pipe_quick.fit(X_train, y_train)
print(f"Quick Ridge baseline:")
print(f"  Train R²: {pipe_quick.score(X_train, y_train):.4f}")
print(f"  Test R²:  {pipe_quick.score(X_test, y_test):.4f}")

Quick Ridge baseline:
  Train R²: 0.6314
  Test R²:  0.6459


In [27]:
# =============================================================================
# SECTION 8: TRAIN AND EVALUATE
# =============================================================================

cv = KFold(n_splits=CONFIG["cv_folds"], shuffle=True, random_state=CONFIG["random_state"])
log_file = "Outputs/experiment_log.xlsx"

results = {}

for name, config in param_grids.items():
    print(f"\n{'='*50}")
    print(f"Training {name}...")
    print(f"{'='*50}")

    start = time.time()

    grid = GridSearchCV(
        estimator=config["pipeline"],
        param_grid=config["params"],
        cv=cv,
        scoring="neg_mean_absolute_error",
        n_jobs=CONFIG["n_jobs"],
        verbose=1,
    )

    grid.fit(X_train, y_train)
    train_time = time.time() - start

    y_pred_train = grid.predict(X_train)
    y_pred_test = grid.predict(X_test)

    results[name] = {
        "best_params": grid.best_params_,
        "train_time": train_time,
        "train_rmse": np.sqrt(mean_squared_error(y_train, y_pred_train)),
        "test_rmse":  np.sqrt(mean_squared_error(y_test, y_pred_test)),
        "train_mae":  mean_absolute_error(y_train, y_pred_train),
        "test_mae":   mean_absolute_error(y_test, y_pred_test),
        "train_r2":   r2_score(y_train, y_pred_train),
        "test_r2":    r2_score(y_test, y_pred_test),
        "model":      grid.best_estimator_,
        "train_mape": np.mean(np.abs((y_train - y_pred_train) / y_train)) * 100,
        "test_mape":  np.mean(np.abs((y_test - y_pred_test) / y_test)) * 100,
    }

    r = results[name]
    print(f"\n  Best params: {r['best_params']}")
    print(f"  Time: {r['train_time']:.1f}s")
    print(f"  Train — RMSE: £{r['train_rmse']:,.0f}, MAE: £{r['train_mae']:,.0f}, R²: {r['train_r2']:.4f}, MAPE: {r['train_mape']:.1f}%")
    print(f"  Test  — RMSE: £{r['test_rmse']:,.0f}, MAE: £{r['test_mae']:,.0f}, R²: {r['test_r2']:.4f}, MAPE: {r['test_mape']:.1f}%")

    # Log immediately after each model
    log_entry = {
        "Timestamp": datetime.now().strftime("%Y-%m-%d %H:%M"),
        "Model": name,
        "Rows": len(df),
        "Train Rows": len(X_train),
        "Test Rows": len(X_test),
        "Features": len(all_features),
        "Train R²": r["train_r2"],
        "Test R²": r["test_r2"],
        "Train RMSE": r["train_rmse"],
        "Test RMSE": r["test_rmse"],
        "Train MAE": r["train_mae"],
        "Test MAE": r["test_mae"],
        "Train MAPE": r["train_mape"],
        "Test MAPE": r["test_mape"],
        "Time (s)": round(r["train_time"], 1),
        "Best Parameters": str(r["best_params"]),
        "Notes": "",
    }

    log_df = pd.DataFrame([log_entry])
    if os.path.exists(log_file):
        with pd.ExcelWriter(
            log_file,
            engine="openpyxl",
            mode="a",
            if_sheet_exists="overlay",
        ) as writer:
            startrow = writer.sheets["Sheet1"].max_row
            log_df.to_excel(
                writer,
                index=False,
                header=False,
                startrow=startrow,
            )
    else:
        log_df.to_excel(log_file, index=False)

    print(f"Logged to {log_file}")


Training Ridge...
Fitting 5 folds for each of 5 candidates, totalling 25 fits

  Best params: {'regressor__alpha': 200.0}
  Time: 6.7s
  Train — RMSE: £169,242, MAE: £96,517, R²: 0.6314, MAPE: 36.8%
  Test  — RMSE: £168,053, MAE: £95,715, R²: 0.6459, MAPE: 37.6%
Logged to Outputs/experiment_log.xlsx

Training XGBoost...
Fitting 5 folds for each of 32 candidates, totalling 160 fits

  Best params: {'regressor__colsample_bytree': 0.8, 'regressor__gamma': 0.2, 'regressor__learning_rate': 0.05, 'regressor__max_depth': 8, 'regressor__min_child_weight': 5, 'regressor__n_estimators': 500, 'regressor__reg_alpha': 0, 'regressor__reg_lambda': 3, 'regressor__subsample': 0.8}
  Time: 489.2s
  Train — RMSE: £59,991, MAE: £38,658, R²: 0.9537, MAPE: 15.1%
  Test  — RMSE: £110,605, MAE: £55,548, R²: 0.8466, MAPE: 19.9%
Logged to Outputs/experiment_log.xlsx

Training RandomForest...
Fitting 5 folds for each of 12 candidates, totalling 60 fits


KeyboardInterrupt: 

In [ ]:
# =============================================================================
# SECTION 9: LOG RESULTS
# =============================================================================

log = pd.read_csv(log_file)
print(log.tail(len(results)).to_string(index=False))

Results logged to Outputs/experiment_log.csv
       timestamp        model    rows    train/test  features  test_R² test_MAE test_MAPE test_RMSE  train_R² time                                                                                                                                                          params  notes
2026-08-05 21:31        Ridge 100,000 80,000/20,000        25   0.6125  £91,273     34.9%  £167,694    0.6263   3s                                                                                                                                     {'regressor__alpha': 100.0}    NaN
2026-08-05 21:31      XGBoost 100,000 80,000/20,000        25   0.8549  £49,329     17.3%  £102,623    0.9874 141s {'regressor__colsample_bytree': 0.8, 'regressor__learning_rate': 0.05, 'regressor__max_depth': 10, 'regressor__n_estimators': 500, 'regressor__subsample': 0.8}    NaN
2026-08-05 21:31 RandomForest 100,000 80,000/20,000        25   0.8284  £55,966     19.9%  £111,602    0.9240